In [ ]:
import requests
r = requests.get("https://proverki.gov.ru/", timeout=20)
print(r.status_code)

In [ ]:
s = requests.Session()
s.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Referer": "https://proverki.gov.ru/portal/public-inspections",
    "Origin": "https://proverki.gov.ru",
})

s.get("https://proverki.gov.ru/portal", timeout=20)          # забрать куки
r = s.get("https://proverki.gov.ru/public/api/inspections/find",
          params={"page": "50,1", "searchString": "7707083893"}, timeout=20)
print(r.status_code, r.text[:300])

In [ ]:
!pip install httpx[http2]

In [ ]:
import httpx
with httpx.Client(http2=True, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Referer": "https://proverki.gov.ru/portal/public-inspections",
    "Origin": "https://proverki.gov.ru"}, timeout=20) as c:
    r = c.get("https://proverki.gov.ru/portal")
    print(r.status_code)

In [ ]:
print(r)

In [ ]:
import httpx
from time import sleep

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Referer": "https://proverki.gov.ru/portal/public-inspections",
}

BASE = "https://proverki.gov.ru/public/api/inspections/find"
INN = "7707083893"

with httpx.Client(http2=True, headers=HEADERS, timeout=20, follow_redirects=True) as c:
    c.get("https://proverki.gov.ru/portal")          # взять куки

    tries = [
        ("raw comma",      f"{BASE}?page=50,1&searchString={INN}", None),
        ("encoded comma",  f"{BASE}?page=50%2C1&searchString={INN}", None),
        ("params dict",    BASE, {"page": "50,1", "searchString": INN}),
        ("reversed pair",  BASE, {"page": "1,50", "searchString": INN}),
        ("page+size",      BASE, {"page": 0, "size": 50, "searchString": INN}),
        ("only search",    BASE, {"searchString": INN}),
    ]

    for name, url, params in tries:
        sleep(2)
        r = c.get(url, params=params)
        body = r.text[:250].replace("\n", " ")
        print(f"{name:15} {r.status_code}  {body}")
        print(dict(r.headers))
        print('-'*80)

raw comma       400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:35 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server-timing': 'dtSInfo;desc="1"'}
------------------------------
encoded comma   400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:37 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server-timing': 'dtSInfo;desc="0", dtRpid;desc="-21741765"'}
------------------------------
params dict     400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:39 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server-timing': 'dtSInfo;desc="1"'}
------------------------------
reversed pair   400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:41 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server-timing': 'dtSInfo;desc="0", dtRpid;desc="1169458953"'}
------------------------------
page+size       400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:43 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server

In [10]:
import requests

# ВАЖНО: Сначала получите cookie, зайдя на главную страницу
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://proverki.gov.ru/",
    "Origin": "https://proverki.gov.ru",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "Connection": "keep-alive",
})

# Заходим на главную, чтобы получить session cookies
session.get("https://proverki.gov.ru/")

# Теперь делаем запрос к API с правильными параметрами
# Скорее всего, page — это номер страницы (с 0 или 1), а size — размер
url = "https://proverki.gov.ru/public/api/inspections/find"
params = {
    "page": 1,            # или 1, попробуйте оба
    "size": 50,           # или другой размер
    "searchString": "7707083893"
}

r = session.get(url, params=params)
print(r.status_code)
print(r.text[:500])

400



тест парсинга новых законов

In [1]:
API = "http://publication.pravo.gov.ru/api"
BLOCKS = ("president", "assembly", "government", "federal_authorities", "subjects")
AUTO_PUBLISH = True   # False — всё через модерацию

KEYWORDS = ("налог", "страхов", "предпринимател", "малого и среднего", "контрол", "надзор", "проверк",
            "маркировк", "контрольно-кассов", "трудов", "персональн", "лиценз", "закупк", "платформ",
            "самозанят", "бухгалтер", "отчетност", "торгов", "общественного питания", "иностранн")

In [2]:
import requests
from datetime import date, datetime, timedelta

API = "http://publication.pravo.gov.ru/api"
day = date.today() - timedelta(days=30)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": API,
    "Origin": API,
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "Connection": "keep-alive",
})

# Заходим на главную, чтобы получить session cookies
session.get(API)

# Теперь делаем запрос к API с правильными параметрами
# Скорее всего, page — это номер страницы (с 0 или 1), а size — размер
url = f'{API}/Documents'
for block in BLOCKS:
    params={
    "Block": block, "PeriodType": "day",
    "Date": day.strftime("%d.%m.%Y"),   # TODO: сверить формат даты на живом API
    "Index": 1}
    r = session.get(url, params=params)
    print('Block: ', block)
    print(r.status_code)
    print(r.text)
    print('-'*50)


Block:  president
200
{"items":[{"eoNumber":"0001202608240009","hasSvg":false,"zipFileLength":null,"publishDateShort":"2026-08-24T00:00:00","complexName":"Указ Президента Российской Федерации от 24.08.2026 № 604\n \"О мерах по обеспечению безопасности объектов критической инфраструктуры Российской Федерации\"","pagesCount":3,"pdfFileLength":212328,"jdRegNumber":null,"jdRegDate":null,"name":"\"О мерах по обеспечению безопасности объектов критической инфраструктуры Российской Федерации\"","number":"604","documentDate":"2026-08-24T00:00:00","signatoryAuthorityId":"225698f1-cfbc-4e42-9caa-32f9f7403211","documentTypeId":"0790e34b-784b-4372-884e-3282622a24bd","title":"Указ Президента Российской Федерации от 24.08.2026 № 604<br /> \"О мерах по обеспечению безопасности объектов критической инфраструктуры Российской Федерации\"","viewDate":"24.08.2026","id":"5710c479-8095-475a-a21e-fe759668b9cd"},{"eoNumber":"0001202608240007","hasSvg":false,"zipFileLength":null,"publishDateShort":"2026-08-24T0

In [3]:
params_single={"eoNumber": '0101202608240005'}
r = session.get(url, params=params_single)
print(r.status_code)
print(r.text)
print('-'*50)

200
{"items":[{"eoNumber":"0101202608240005","hasSvg":false,"zipFileLength":null,"publishDateShort":"2026-08-24T00:00:00","complexName":"Приказ Министерства труда и социального развития Республики Адыгея  от 20.08.2026 № 1301\n \"О внесении изменений в приказ Министерства труда и социального развития Республики Адыгея от 10 июля 2026 года № 1046 \"Об утверждении Административного регламента предоставления государственной услуги \"Признание гражданина нуждающимся в социальном обслуживании\"","pagesCount":2,"pdfFileLength":137103,"jdRegNumber":"26-358","jdRegDate":"2026-08-24T00:00:00","name":"О внесении изменений в приказ Министерства труда и социального развития Республики Адыгея от 10 июля 2026 года № 1046 «Об утверждении Административного регламента предоставления государственной услуги «Признание гражданина нуждающимся в социальном обслуживании»","number":"1301","documentDate":"2026-08-20T00:00:00","signatoryAuthorityId":"ce4040ef-934c-4fc9-8768-75ee1600e918","documentTypeId":"2dddb

In [ ]:
import httpx

async def fetch_published(client: httpx.AsyncClient, day: date) -> list[dict]:
    out: list[dict] = []
    for block in BLOCKS:
        page = 1
        while True:
            r = await client.get(f"{API}/Documents", params={
                "Block": block, "PeriodType": "day",
                "Date": day.strftime("%d.%m.%Y"),   # TODO: сверить формат даты на живом API
                "PageSize": 200, "Index": page})
            r.raise_for_status()
            data = r.json()
            out += [{**it, "block": block} for it in data.get("items", [])]
            print(out)
            if page >= data.get("pagesTotalCount", 1):
                break
            page += 1
    # return out

In [7]:
day = date.today() - timedelta(days=1)
async with httpx.AsyncClient(timeout=30) as client:
    print(await fetch_published(client, day))

[]
[]
[{'eoNumber': '0001202609220038', 'hasSvg': False, 'zipFileLength': None, 'publishDateShort': '2026-09-22T00:00:00', 'complexName': 'Постановление Правительства Российской Федерации от 22.09.2026 № 1215\n "О внесении изменения в постановление Правительства Российской Федерации от 15 июля 2009 г. № 602"', 'pagesCount': 1, 'pdfFileLength': 228297, 'jdRegNumber': None, 'jdRegDate': None, 'name': 'О внесении изменения в постановление Правительства Российской Федерации от 15 июля 2009 г. № 602', 'number': '1215', 'documentDate': '2026-09-22T00:00:00', 'signatoryAuthorityId': '8005d8c9-4b6d-48d3-861a-2a37e69fccb3', 'documentTypeId': 'fd5a8766-f6fd-4ac2-8fd9-66f414d314ac', 'title': 'Постановление Правительства Российской Федерации от 22.09.2026 № 1215<br /> "О внесении изменения в постановление Правительства Российской Федерации от 15 июля 2009 г. № 602"', 'viewDate': '22.09.2026', 'id': '4d9a2af2-1897-4e94-81fa-41e1ca1166d1', 'block': 'government'}, {'eoNumber': '0001202609220035', 'ha